# Import Librares

In [3]:
import sqlite3
import csv
import pandas as pd

# Define Data Path

In [2]:
data_path = '../MIMIC_Data/physionet.org/files/mimiciii/1.4/1'

# Create a SQL db given MIMIC csv files

In [3]:
conn = sqlite3.connect('mimic_database.db')
file_names = [
    "ADMISSIONS",
    "CALLOUT",
    "CAREGIVERS",
    "CHARTEVENTS",
    "CPTEVENTS",
    "D_CPT",
    "D_ICD_DIAGNOSES",
    "D_ICD_PROCEDURES",
    "D_ITEMS",
    "D_LABITEMS",
    "DATETIMEEVENTS",
    "DIAGNOSES_ICD",
    "DRGCODES",
    "ICUSTAYS",
    "INPUTEVENTS_CV",
    "INPUTEVENTS_MV",
    "LABEVENTS",
    "MICROBIOLOGYEVENTS",
    "NOTEEVENTS",
    "OUTPUTEVENTS",
    "PATIENTS",
    "PRESCRIPTIONS",
    "PROCEDUREEVENTS_MV",
    "PROCEDURES_ICD",
    "SERVICES",
    "TRANSFERS"
]
for file in file_names:
    for chunk in pd.read_csv(f"{data_path}/{file}.csv", chunksize=100000):
        chunk.to_sql(file, conn, index=False, if_exists='append')

conn.close()

C:\Users\patel\AppData\Local\Temp\ipykernel_26028\2926986507.py:31: DtypeWarning: Columns (13) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(f"{data_path}/{file}.csv", chunksize=100000):
C:\Users\patel\AppData\Local\Temp\ipykernel_26028\2926986507.py:31: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(f"{data_path}/{file}.csv", chunksize=100000):
C:\Users\patel\AppData\Local\Temp\ipykernel_26028\2926986507.py:31: DtypeWarning: Columns (5,7) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(f"{data_path}/{file}.csv", chunksize=100000):
C:\Users\patel\AppData\Local\Temp\ipykernel_26028\2926986507.py:31: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(f"{data_path}/{file}.csv", chunksize=100000):
C:\Users\patel\AppData\Local\Temp

# Create DB Connection

In [4]:
conn = sqlite3.connect('mimic_database.db')
cur = conn.cursor()

# Create SQL Queries

# Demographic Analysis

## Query 1

Patient analysis. Determine the number of male and female patients.

In [5]:
query = '''
            SELECT GENDER, COUNT(*)
            FROM PATIENTS
            GROUP BY GENDER;
        '''
cur.execute(query)
rows = cur.fetchall()
print(rows)

[('F', 20399), ('M', 26121)]


## Query 2

Patient analysis. Determine the age distribution of the patients.

In [6]:
query = '''
            SELECT 
                CASE 
                    WHEN strftime('%Y', DOD) - strftime('%Y', DOB) < 18 THEN '0-17'
                    WHEN strftime('%Y', DOD) - strftime('%Y', DOB) BETWEEN 18 AND 34 THEN '18-34'
                    WHEN strftime('%Y', DOD) - strftime('%Y', DOB) BETWEEN 35 AND 49 THEN '35-49'
                    WHEN strftime('%Y', DOD) - strftime('%Y', DOB) BETWEEN 50 AND 64 THEN '50-64'
                    WHEN strftime('%Y', DOD) - strftime('%Y', DOB) BETWEEN 65 AND 79 THEN '65-79'
                    WHEN DOD IS NOT NULL THEN '80+'
                    ELSE 'Alive'
                END AS age_group,
                COUNT(*) AS patient_count
            FROM PATIENTS
            GROUP BY age_group
            ORDER BY age_group;
        '''
cur.execute(query)
rows = cur.fetchall()
print(rows)

[('0-17', 74), ('18-34', 278), ('35-49', 1010), ('50-64', 3023), ('65-79', 5146), ('80+', 6228), ('Alive', 30761)]


## Query 3

Determine the different types of Religions patient practice, and their counts.

In [7]:
query = '''
            WITH first_admissions AS (
                SELECT SUBJECT_ID, RELIGION
                FROM ADMISSIONS
                WHERE ADMITTIME = (SELECT MIN(ADMITTIME) FROM ADMISSIONS a2 WHERE a2.SUBJECT_ID = ADMISSIONS.SUBJECT_ID)
            )
            SELECT 
                RELIGION, 
                COUNT(*) AS patient_count
            FROM first_admissions
            GROUP BY RELIGION
            ORDER BY patient_count DESC;
        '''
cur.execute(query)
rows = cur.fetchall()
print(rows)

[('CATHOLIC', 15659), ('NOT SPECIFIED', 9550), ('UNOBTAINABLE', 7711), ('PROTESTANT QUAKER', 5117), ('JEWISH', 3833), ('OTHER', 2104), ('EPISCOPALIAN', 589), (None, 443), ('CHRISTIAN SCIENTIST', 360), ('GREEK ORTHODOX', 323), ('BUDDHIST', 195), ('MUSLIM', 157), ('UNITARIAN-UNIVERSALIST', 104), ("JEHOVAH'S WITNESS", 104), ('HINDU', 101), ('ROMANIAN EAST. ORTH', 66), ('7TH DAY ADVENTIST', 57), ('BAPTIST', 25), ('HEBREW', 15), ('METHODIST', 6), ('LUTHERAN', 1)]


### Query 3.2 - Alternative Method

In [8]:
query = '''
            SELECT 
                a.RELIGION, 
                COUNT(*) AS patient_count
            FROM ADMISSIONS a
            JOIN (
                SELECT SUBJECT_ID, MIN(ADMITTIME) AS first_admit 
                FROM ADMISSIONS 
                GROUP BY SUBJECT_ID
            ) fa ON a.SUBJECT_ID = fa.SUBJECT_ID AND a.ADMITTIME = fa.first_admit
            GROUP BY a.RELIGION
            ORDER BY patient_count DESC;
        '''
cur.execute(query)
rows = cur.fetchall()
print(rows)

[('CATHOLIC', 15659), ('NOT SPECIFIED', 9550), ('UNOBTAINABLE', 7711), ('PROTESTANT QUAKER', 5117), ('JEWISH', 3833), ('OTHER', 2104), ('EPISCOPALIAN', 589), (None, 443), ('CHRISTIAN SCIENTIST', 360), ('GREEK ORTHODOX', 323), ('BUDDHIST', 195), ('MUSLIM', 157), ('UNITARIAN-UNIVERSALIST', 104), ("JEHOVAH'S WITNESS", 104), ('HINDU', 101), ('ROMANIAN EAST. ORTH', 66), ('7TH DAY ADVENTIST', 57), ('BAPTIST', 25), ('HEBREW', 15), ('METHODIST', 6), ('LUTHERAN', 1)]


# Query 4

Determine the count of patient's ethnicities

In [13]:
query = '''
    SELECT 
        a.ETHNICITY, 
        COUNT(*) AS patient_count
    FROM ADMISSIONS a
    JOIN (
        SELECT SUBJECT_ID, MIN(ADMITTIME) AS first_admit 
        FROM ADMISSIONS 
        GROUP BY SUBJECT_ID
    ) fa ON a.SUBJECT_ID = fa.SUBJECT_ID AND a.ADMITTIME = fa.first_admit
    GROUP BY a.ETHNICITY
    ORDER BY patient_count DESC;
'''
cur.execute(query)
rows = cur.fetchall()
print(rows)


[('WHITE', 32074), ('UNKNOWN/NOT SPECIFIED', 4236), ('BLACK/AFRICAN AMERICAN', 3585), ('HISPANIC OR LATINO', 1350), ('ASIAN', 1304), ('OTHER', 1256), ('UNABLE TO OBTAIN', 792), ('PATIENT DECLINED TO ANSWER', 498), ('ASIAN - CHINESE', 223), ('BLACK/CAPE VERDEAN', 159), ('HISPANIC/LATINO - PUERTO RICAN', 146), ('MULTI RACE ETHNICITY', 111), ('WHITE - RUSSIAN', 105), ('BLACK/HAITIAN', 71), ('WHITE - OTHER EUROPEAN', 69), ('HISPANIC/LATINO - DOMINICAN', 60), ('ASIAN - ASIAN INDIAN', 57), ('AMERICAN INDIAN/ALASKA NATIVE', 45), ('WHITE - BRAZILIAN', 42), ('ASIAN - VIETNAMESE', 41), ('PORTUGUESE', 36), ('BLACK/AFRICAN', 32), ('MIDDLE EASTERN', 28), ('HISPANIC/LATINO - GUATEMALAN', 25), ('WHITE - EASTERN EUROPEAN', 22), ('NATIVE HAWAIIAN OR OTHER PACIFIC ISLANDER', 15), ('HISPANIC/LATINO - CUBAN', 15), ('ASIAN - OTHER', 15), ('ASIAN - FILIPINO', 15), ('HISPANIC/LATINO - SALVADORAN', 14), ('HISPANIC/LATINO - MEXICAN', 11), ('ASIAN - KOREAN', 11), ('ASIAN - CAMBODIAN', 10), ('HISPANIC/LATINO - C

## Query 5

Lab events analysis. Get the lab event's "ITEMID" sorted by the number of times the "FLAG" was "abnormal"

In [6]:
query = '''
            SELECT 
                ITEMID, 
                COUNT(*) AS abnormal_count
            FROM LABEVENTS
            WHERE FLAG = 'abnormal'
            GROUP BY ITEMID
            ORDER BY abnormal_count DESC;
        '''
cur.execute(query)
rows = cur.fetchall()
print(rows)


[(51221, 783689), (51279, 673592), (51222, 667697), (50931, 508914), (51006, 442789), (51277, 346186), (51274, 337209), (50821, 325988), (50912, 321092), (51301, 320247), (51265, 287204), (50893, 268662), (51275, 235026), (51237, 222487), (50970, 219002), (50902, 217537), (50882, 214703), (51248, 208490), (50818, 200752), (50820, 197677), (51249, 157729), (50809, 154439), (50983, 130416), (50804, 129656), (51250, 129401), (51256, 124801), (51244, 115710), (50808, 111158), (50878, 99614), (50971, 95976), (50863, 94076), (50862, 91552), (50861, 89439), (50811, 80401), (50885, 78440), (50813, 77982), (50960, 67117), (50910, 62225), (51003, 59242), (50954, 58100), (50868, 48993), (51493, 38514), (51009, 38333), (50822, 35541), (51516, 23470), (51254, 22654), (50956, 21982), (51200, 21944), (50911, 21896), (51214, 20262), (50883, 19442), (50824, 18770), (51251, 17279), (50867, 17271), (51143, 15623), (51218, 15344), (51257, 14006), (50967, 12688), (50806, 12627), (51144, 12051), (51255, 115

## Query 6

From Query 5, we see that the top lab event that has the highest number of abnormal tests has an ITEMID = 51221

Using the D_LABITEMS table, determine the LABEL associated with ITEMID = 51221

In [7]:
query = '''
            SELECT 
                LABEL
            FROM D_LABITEMS
            WHERE ITEMID = 51221;
        '''
cur.execute(query)
rows = cur.fetchall()
print(rows)

[('Hematocrit',)]


## Query 7

Get the SUBJECT ID of the patients who tested abnormal for the lab test with ITEMID = 51221

In [14]:
query = '''
    SELECT DISTINCT SUBJECT_ID
    FROM LABEVENTS
    WHERE ITEMID = 51221
    AND FLAG = 'abnormal';
'''
cur.execute(query)
rows = cur.fetchall()
print(rows)

[(3,), (2,), (4,), (6,), (7,), (8,), (11,), (9,), (12,), (13,), (17,), (16,), (19,), (20,), (21,), (18,), (22,), (23,), (36,), (26,), (35,), (32,), (27,), (28,), (30,), (33,), (31,), (34,), (25,), (38,), (41,), (42,), (44,), (43,), (64,), (45,), (61,), (52,), (55,), (46,), (59,), (62,), (49,), (54,), (67,), (68,), (37,), (56,), (57,), (80,), (77,), (84,), (93,), (94,), (85,), (69,), (91,), (72,), (81,), (73,), (78,), (71,), (86,), (83,), (74,), (79,), (75,), (107,), (88,), (97,), (100,), (105,), (96,), (92,), (99,), (109,), (101,), (98,), (103,), (108,), (95,), (87,), (115,), (114,), (106,), (113,), (129,), (130,), (112,), (117,), (124,), (125,), (110,), (118,), (119,), (123,), (111,), (133,), (137,), (138,), (141,), (135,), (126,), (132,), (134,), (143,), (145,), (127,), (144,), (131,), (136,), (139,), (140,), (142,), (146,), (148,), (149,), (152,), (156,), (147,), (157,), (155,), (151,), (153,), (154,), (161,), (165,), (150,), (171,), (159,), (158,), (160,), (169,), (174,), (175,), (

## Query 8

Get the gender distribution for the patients who tested abnormal for lab test with ITEMID = 51221

In [8]:
query = '''
            SELECT 
                p.GENDER, 
                COUNT(*) AS patient_count
            FROM PATIENTS p
            JOIN LABEVENTS l ON p.SUBJECT_ID = l.SUBJECT_ID
            WHERE l.ITEMID = 51221
            AND l.FLAG = 'abnormal'
            GROUP BY p.GENDER;
        '''
cur.execute(query)
rows = cur.fetchall()
print(rows)

[('F', 330697), ('M', 452992)]


# Query 9

Get the count of the drugs that were prescribed to the patients who were abnormal for the lab test "Hematocrit" with ITEMID = 51221

In [24]:
query = '''
            SELECT 
                p.DRUG, 
                COUNT(*) AS prescription_count
            FROM PRESCRIPTIONS p
            JOIN LABEVENTS l ON p.SUBJECT_ID = l.SUBJECT_ID
            WHERE l.ITEMID = 51221
            AND l.FLAG = 'abnormal'
            GROUP BY p.DRUG
            ORDER BY prescription_count DESC;
        '''
cur.execute(query)
rows = cur.fetchall()
print(rows)

[('Insulin', 8008566), ('Potassium Chloride', 7982974), ('D5W', 7523767), ('NS', 6986850), ('Furosemide', 6649484), ('0.9% Sodium Chloride', 5152816), ('Iso-Osmotic Dextrose', 4438734), ('Magnesium Sulfate', 3881055), ('SW', 3389379), ('5% Dextrose', 3305286), ('Metoprolol', 3293109), ('Sodium Chloride 0.9%  Flush', 2859511), ('Acetaminophen', 2751827), ('Lorazepam', 2699593), ('Morphine Sulfate', 2475809), ('Calcium Gluconate', 2242909), ('Vancomycin', 2119770), ('Heparin', 2078598), ('Metoprolol Tartrate', 1979423), ('Warfarin', 1941867), ('HYDROmorphone (Dilaudid)', 1795354), ('Heparin Sodium', 1720842), ('Tacrolimus', 1598630), ('Pantoprazole', 1562473), ('Fentanyl Citrate', 1510102), ('Docusate Sodium', 1465404), ('Bisacodyl', 1430453), ('Vancomycin HCl', 1391501), ('Vial', 1373362), ('LR', 1373106), ('Propofol', 1233434), ('Sodium Bicarbonate', 1122322), ('Senna', 1070843), ('Aspirin', 1010024), ('Levofloxacin', 965010), ('Ondansetron', 930058), ('Bag', 929393), ('Albuterol 0.083

## Query 10

Look at the religion distrbution for patients who were abnormal for the lab test "Hematocrit" with ITEMID = 51221

In [10]:
query = '''
            SELECT 
                a.RELIGION, 
                COUNT(*) AS patient_count
            FROM ADMISSIONS a
            JOIN (
                SELECT SUBJECT_ID, MIN(ADMITTIME) AS first_admit FROM ADMISSIONS GROUP BY SUBJECT_ID
            ) fa ON a.SUBJECT_ID = fa.SUBJECT_ID AND a.ADMITTIME = fa.first_admit
            JOIN LABEVENTS l ON a.SUBJECT_ID = l.SUBJECT_ID
            WHERE l.ITEMID = 51221
            AND l.FLAG = 'abnormal'
            GROUP BY a.RELIGION
            ORDER BY patient_count DESC;
        '''
cur.execute(query)
rows = cur.fetchall()
print(rows)


[('CATHOLIC', 301244), ('NOT SPECIFIED', 142428), ('PROTESTANT QUAKER', 109532), ('JEWISH', 80191), ('UNOBTAINABLE', 68851), ('OTHER', 37810), ('EPISCOPALIAN', 11159), ('GREEK ORTHODOX', 7173), (None, 5107), ('CHRISTIAN SCIENTIST', 4791), ('MUSLIM', 3572), ('BUDDHIST', 3570), ("JEHOVAH'S WITNESS", 1947), ('UNITARIAN-UNIVERSALIST', 1737), ('7TH DAY ADVENTIST', 1407), ('ROMANIAN EAST. ORTH', 1103), ('HINDU', 965), ('BAPTIST', 625), ('HEBREW', 309), ('METHODIST', 162), ('LUTHERAN', 6)]


## Query 11

Look at the ethnicity distrbution for patients who were abnormal for the lab test "Hematocrit" with ITEMID = 51221

In [11]:
query = '''
    SELECT 
        a.ETHNICITY, 
        COUNT(*) AS patient_count
    FROM ADMISSIONS a
    JOIN (
        SELECT SUBJECT_ID, MIN(ADMITTIME) AS first_admit FROM ADMISSIONS GROUP BY SUBJECT_ID
    ) fa ON a.SUBJECT_ID = fa.SUBJECT_ID AND a.ADMITTIME = fa.first_admit
    JOIN LABEVENTS l ON a.SUBJECT_ID = l.SUBJECT_ID
    WHERE l.ITEMID = 51221
    AND l.FLAG = 'abnormal'
    GROUP BY a.ETHNICITY
    ORDER BY patient_count DESC;
'''
cur.execute(query)
rows = cur.fetchall()
print(rows)

[('WHITE', 560923), ('BLACK/AFRICAN AMERICAN', 77278), ('UNKNOWN/NOT SPECIFIED', 55664), ('HISPANIC OR LATINO', 21366), ('OTHER', 15162), ('ASIAN', 12486), ('UNABLE TO OBTAIN', 10288), ('PATIENT DECLINED TO ANSWER', 6369), ('ASIAN - CHINESE', 3144), ('HISPANIC/LATINO - PUERTO RICAN', 2986), ('BLACK/CAPE VERDEAN', 2387), ('MULTI RACE ETHNICITY', 1859), ('WHITE - RUSSIAN', 1512), ('ASIAN - ASIAN INDIAN', 1216), ('PORTUGUESE', 1205), ('BLACK/HAITIAN', 1161), ('WHITE - BRAZILIAN', 1078), ('WHITE - OTHER EUROPEAN', 1047), ('HISPANIC/LATINO - DOMINICAN', 934), ('MIDDLE EASTERN', 814), ('ASIAN - VIETNAMESE', 593), ('AMERICAN INDIAN/ALASKA NATIVE', 484), ('HISPANIC/LATINO - CUBAN', 417), ('ASIAN - CAMBODIAN', 415), ('BLACK/AFRICAN', 407), ('HISPANIC/LATINO - GUATEMALAN', 376), ('WHITE - EASTERN EUROPEAN', 364), ('ASIAN - OTHER', 287), ('ASIAN - FILIPINO', 229), ('HISPANIC/LATINO - SALVADORAN', 219), ('ASIAN - KOREAN', 140), ('NATIVE HAWAIIAN OR OTHER PACIFIC ISLANDER', 122), ('HISPANIC/LATINO 

## Query 12

Look at the insurance distribution for patients who were abnormal for the lab test "Hematocrit" with ITEMID = 51221

In [12]:
query = '''
    SELECT 
        a.INSURANCE, 
        COUNT(*) AS patient_count
    FROM ADMISSIONS a
    JOIN (
        SELECT SUBJECT_ID, MIN(ADMITTIME) AS first_admit FROM ADMISSIONS GROUP BY SUBJECT_ID
    ) fa ON a.SUBJECT_ID = fa.SUBJECT_ID AND a.ADMITTIME = fa.first_admit
    JOIN LABEVENTS l ON a.SUBJECT_ID = l.SUBJECT_ID
    WHERE l.ITEMID = 51221
    AND l.FLAG = 'abnormal'
    GROUP BY a.INSURANCE
    ORDER BY patient_count DESC;
'''
cur.execute(query)
rows = cur.fetchall()
print(rows)


[('Medicare', 422838), ('Private', 259843), ('Medicaid', 75413), ('Government', 20293), ('Self Pay', 5302)]


## Query 13

Look at the admission type for patients who were abnormal for the lab test "Hematocrit" with ITEMID = 51221


In [15]:
query = '''
    SELECT 
        a.ADMISSION_TYPE, 
        COUNT(*) AS patient_count
    FROM ADMISSIONS a
    JOIN (
        SELECT SUBJECT_ID, MIN(ADMITTIME) AS first_admit FROM ADMISSIONS GROUP BY SUBJECT_ID
    ) fa ON a.SUBJECT_ID = fa.SUBJECT_ID AND a.ADMITTIME = fa.first_admit
    JOIN LABEVENTS l ON a.SUBJECT_ID = l.SUBJECT_ID
    WHERE l.ITEMID = 51221
    AND l.FLAG = 'abnormal'
    GROUP BY a.ADMISSION_TYPE
    ORDER BY patient_count DESC;
'''
cur.execute(query)
rows = cur.fetchall()
print(rows)


[('EMERGENCY', 636664), ('ELECTIVE', 113822), ('URGENT', 26194), ('NEWBORN', 7009)]


## Query 14

Determine the average length of stay for the admissions where the patient tested abnormal for the lab test "Hematocrit" with ITEMID = 51221 (can split by admission type (EMERGENCY, URGENT, ELECTIVE, etc.))

In [17]:
query = '''
    SELECT 
        AVG(julianday(a.DISCHTIME) - julianday(a.ADMITTIME)) AS overall_avg_length_of_stay
    FROM ADMISSIONS a
    JOIN LABEVENTS l ON a.SUBJECT_ID = l.SUBJECT_ID AND a.HADM_ID = l.HADM_ID
    WHERE l.ITEMID = 51221
    AND l.FLAG = 'abnormal';
'''
cur.execute(query)
rows = cur.fetchall()
print(rows)

[(21.155649549779117,)]


The unit of the average length of stay is days.

## Query 15

Determine the top 10 diagnosis for the admissions where the patient tested abnormal for the lab test "Hematocrit" with ITEMID = 51221

In [18]:
query = '''
    SELECT 
        d.ICD9_CODE, 
        COUNT(*) AS diagnosis_count
    FROM DIAGNOSES_ICD d
    JOIN LABEVENTS l ON d.SUBJECT_ID = l.SUBJECT_ID AND d.HADM_ID = l.HADM_ID
    WHERE l.ITEMID = 51221
    AND l.FLAG = 'abnormal'
    GROUP BY d.ICD9_CODE
    ORDER BY diagnosis_count DESC
    LIMIT 10;
'''
cur.execute(query)
rows = cur.fetchall()
print(rows)

[('4019', 226943), ('4280', 188480), ('42731', 185664), ('5849', 148793), ('51881', 142392), ('41401', 140566), ('5990', 111389), ('25000', 108999), ('2851', 94504), ('2724', 91481)]


## Query 16

Get the short title and the long title of the diagnosis with the highest count for patients who tested abnormal for lab test with ITEMID = 51221

In [19]:
query = '''
    SELECT SHORT_TITLE, LONG_TITLE
    FROM D_ICD_DIAGNOSES
    WHERE ICD9_CODE = '4019';
'''
cur.execute(query)
rows = cur.fetchall()
print(rows)

[('Hypertension NOS', 'Unspecified essential hypertension')]


## Query 17

Determine the types of ICUs

In [7]:
query = '''
    SELECT FIRST_CAREUNIT AS ICU_TYPE FROM ICUSTAYS
    UNION
    SELECT LAST_CAREUNIT FROM ICUSTAYS;
'''
cur.execute(query)
rows = cur.fetchall()
print(rows)

[('CCU',), ('CSRU',), ('MICU',), ('NICU',), ('SICU',), ('TSICU',)]


## Query 18

Determine the counts of "FIRST_CAREUNIT" for ICU stays of patients

In [8]:
query = '''
    SELECT FIRST_CAREUNIT, COUNT(*) AS unit_count
    FROM ICUSTAYS
    GROUP BY FIRST_CAREUNIT
    ORDER BY unit_count DESC;
'''
cur.execute(query)
rows = cur.fetchall()
print(rows)

[('MICU', 21088), ('CSRU', 9312), ('SICU', 8891), ('NICU', 8100), ('CCU', 7726), ('TSICU', 6415)]


## Query 19

From Query 18, for the "FIRST_CAREUNIT" with the largest count, determine the length-of-stay distribution.

In [12]:
query = '''
    WITH Most_Common_Unit AS (
        -- Get the most frequent FIRST_CAREUNIT
        SELECT FIRST_CAREUNIT
        FROM ICUSTAYS
        GROUP BY FIRST_CAREUNIT
        ORDER BY COUNT(*) DESC
        LIMIT 1
    )
    SELECT 
        CASE 
            WHEN LOS < 1 THEN '0-1 days'
            WHEN LOS BETWEEN 1 AND 3 THEN '1-3 days'
            WHEN LOS BETWEEN 4 AND 7 THEN '4-7 days'
            WHEN LOS BETWEEN 8 AND 14 THEN '8-14 days'
            ELSE '15+ days'
        END AS LOS_Category,
        COUNT(*) AS patient_count
    FROM ICUSTAYS
    WHERE FIRST_CAREUNIT = (SELECT FIRST_CAREUNIT FROM Most_Common_Unit)
    GROUP BY LOS_Category
    ORDER BY patient_count DESC;
'''
cur.execute(query)
rows = cur.fetchall()
print(rows)

[('1-3 days', 10119), ('0-1 days', 3539), ('15+ days', 3460), ('4-7 days', 2615), ('8-14 days', 1355)]


# Close DB Connection

In [ ]:
conn.close()